# 03 — Predictive Experiment: Basket Completion via Association Graph

Evaluates three recommenders on the basket completion task:
- **Popularity**: context-unaware baseline (global product frequency)
- **LocalAgg**: average direct confidence weights from observed items
- **PPR**: Personalized PageRank over the confidence graph

**Prerequisite:** run `python scripts/run_experiment.py` to generate `outputs/results/`.

## 0. Setup

In [ ]:
import sys, os, json
sys.path.insert(0, '..')

import numpy as np
import scipy.sparse as sp
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr

from src.graph_builder import load_graph

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 120, 'figure.figsize': (10, 5)})

RESULTS_PATH   = '../outputs/results/experiment_results.parquet'
HIT_RATES_PATH = '../outputs/results/product_hit_rates.parquet'
GRAPH_DIR      = '../outputs/graphs'
TEST_DATA      = '../data/processed/test.parquet'
PARAMS_PATH    = '../outputs/results/params.json'

METHOD_LABELS = {'popularity': 'Popularity', 'local_agg': 'LocalAgg', 'ppr': 'PPR'}
METHOD_ORDER  = ['popularity', 'local_agg', 'ppr']

N_OBS_FRACTIONS = [0.25, 0.5, 0.75]
FRAC_LABELS     = {0.25: '25%', 0.5: '50%', 0.75: '75%'}
K_VALUES        = [5, 10, 20]

## 1. Load Results

In [ ]:
with open(PARAMS_PATH) as f:
    params = json.load(f)
print('Experiment params:')
for key, val in params.items():
    print(f'  {key}: {val}')

results = pd.read_parquet(RESULTS_PATH)
print(f'\nShape: {results.shape}')
print(f'Methods:   {sorted(results["method"].unique())}')
print(f'n_obs:     {sorted(results["n_obs"].unique())}')
print(f'k values:  {sorted(results["k"].unique())}')
print(f'Baskets:   {results["basket_id"].nunique():,}')
results.head()

## 2. Table: Mean Precision & Recall at n_obs = 50%

In [ ]:
# Two-step aggregation: repetitions → per-basket mean → grand mean ± std
ref_frac = 0.5

at_half = results[results['n_obs'] == ref_frac].copy()

per_basket = (
    at_half
    .groupby(['basket_id', 'method', 'k'])[['precision', 'recall']]
    .mean()
    .reset_index()
)

summary = (
    per_basket
    .groupby(['method', 'k'])[['precision', 'recall']]
    .agg(['mean', 'std'])
    .round(4)
)
summary.columns = ['precision_mean', 'precision_std', 'recall_mean', 'recall_std']
summary.index = summary.index.set_levels(
    [METHOD_LABELS.get(m, m) for m in summary.index.get_level_values('method').unique()],
    level='method',
)
print('Mean ± Std across baskets at n_obs = 50% of basket observed')
display(summary)

## 3. Line Chart: Recall@k vs Observed Fraction

In [ ]:
# Average over repetitions per basket, then over baskets
per_basket_recall = (
    results
    .groupby(['basket_id', 'n_obs', 'method', 'k'])['recall']
    .mean()
    .reset_index()
)
agg_recall = (
    per_basket_recall
    .groupby(['n_obs', 'method', 'k'])['recall']
    .agg(['mean', 'std'])
    .reset_index()
)

fig, axes = plt.subplots(1, len(K_VALUES), figsize=(5 * len(K_VALUES), 5), sharey=False)

for ax, k_val in zip(axes, K_VALUES):
    sub = agg_recall[agg_recall['k'] == k_val]
    for method in METHOD_ORDER:
        group = sub[sub['method'] == method].sort_values('n_obs')
        label = METHOD_LABELS.get(method, method)
        ax.plot(group['n_obs'], group['mean'], marker='o', markersize=7, label=label)
        ax.fill_between(
            group['n_obs'],
            (group['mean'] - group['std']).clip(lower=0),
            group['mean'] + group['std'],
            alpha=0.15,
        )
    ax.set_xticks(N_OBS_FRACTIONS)
    ax.set_xticklabels([FRAC_LABELS[f] for f in N_OBS_FRACTIONS])
    ax.set_xlabel('Fraction of basket observed')
    ax.set_ylabel(f'Recall@{k_val}')
    ax.set_title(f'Recall@{k_val}')
    if ax == axes[0]:
        ax.legend(title='Method')

fig.suptitle('Recall@k vs Observed Fraction of Basket', y=1.02)
fig.text(0.99, 0.01, 'Source: experiment_results.parquet', ha='right', fontsize=8, color='gray')
plt.tight_layout()
plt.show()

## 4. Complete Metrics: All Fractions × k × Method

In [ ]:
## 5. Graph Structure: In-Strength and Out-Strength

## 4. Graph Structure: In-Strength and Out-Strength

In [ ]:
## 6. Per-Product Hit Rates

## 5. Per-Product Hit Rates

In [ ]:
## 7. Scatter: Hit Rate vs In-Strength

## 6. Scatter: Hit Rate vs In-Strength

In [ ]:
## 8. Scatter: Hit Rate vs Out-Strength

## 7. Scatter: Hit Rate vs Out-Strength

In [ ]:
## 9. Histogram: Distribution of Test Basket Sizes

test_df = pd.read_parquet(TEST_DATA, columns=['basket_id', 'product_idx'])
basket_sizes = test_df.groupby('basket_id')['product_idx'].count()

min_bs = params.get('min_basket_size', 5)
max_bs = params.get('max_basket_size', None)

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(basket_sizes, bins=range(1, min(100, int(basket_sizes.quantile(0.99))) + 2),
        edgecolor='white', linewidth=0.5, color='steelblue', alpha=0.8)
ax.axvline(min_bs, color='tab:green', linestyle='--', linewidth=1.5,
           label=f'min_basket_size = {min_bs}')
if max_bs:
    ax.axvline(max_bs, color='tab:red', linestyle='--', linewidth=1.5,
               label=f'max_basket_size = {max_bs}')
ax.axvline(basket_sizes.median(), color='tab:orange', linestyle=':', linewidth=1.5,
           label=f'Median = {basket_sizes.median():.0f}')
ax.set_xlabel('Basket size (# products)')
ax.set_ylabel('Number of baskets')
ax.set_title('Distribution of Test Basket Sizes')
ax.set_xlim(0, max(50, int(basket_sizes.quantile(0.99)) + 2))
ax.legend()
fig.text(0.99, 0.01, 'Source: test.parquet', ha='right', fontsize=8, color='gray')
plt.tight_layout()
plt.show()

eligible = basket_sizes[
    (basket_sizes >= min_bs) & (basket_sizes <= (max_bs if max_bs else float('inf')))
]
print(f'Total test baskets:                     {len(basket_sizes):,}')
print(f'Evaluated (size {min_bs}–{max_bs or "∞"}): {len(eligible):,} ({100 * len(eligible) / len(basket_sizes):.1f}%)')
print(f'Median size (all):                      {basket_sizes.median():.0f}')
print(f'Median size (evaluated):                {eligible.median():.0f}')

In [ ]:
## 10. Relative Improvement over Popularity Baseline

pop_recall = (
    results[results['method'] == 'popularity']
    .groupby(['basket_id', 'n_obs', 'k'])['recall']
    .mean()
    .reset_index()
    .rename(columns={'recall': 'recall_pop'})
)

method_recall = (
    results
    .groupby(['basket_id', 'n_obs', 'method', 'k'])['recall']
    .mean()
    .reset_index()
)

merged = method_recall.merge(pop_recall, on=['basket_id', 'n_obs', 'k'])
merged['recall_gain'] = merged['recall'] - merged['recall_pop']

gain_agg = (
    merged[merged['method'] != 'popularity']
    .groupby(['n_obs', 'method', 'k'])['recall_gain']
    .mean()
    .reset_index()
)
gain_agg['frac_label']   = gain_agg['n_obs'].map(FRAC_LABELS)
gain_agg['method_label'] = gain_agg['method'].map(METHOD_LABELS)

non_pop_methods = [m for m in METHOD_ORDER if m != 'popularity']

fig, axes = plt.subplots(1, len(K_VALUES), figsize=(5 * len(K_VALUES), 5), sharey=True)
x = np.arange(len(N_OBS_FRACTIONS))
width = 0.35

for ax, k_val in zip(axes, K_VALUES):
    sub = gain_agg[gain_agg['k'] == k_val]
    for i, method in enumerate(non_pop_methods):
        grp = sub[sub['method'] == method].sort_values('n_obs')
        offset = (i - len(non_pop_methods) / 2 + 0.5) * width
        ax.bar(x + offset, grp['recall_gain'].values, width, label=METHOD_LABELS[method])
    ax.set_xticks(x)
    ax.set_xticklabels([FRAC_LABELS[f] for f in N_OBS_FRACTIONS])
    ax.set_xlabel('Fraction of basket observed')
    ax.set_ylabel('Recall gain over Popularity' if ax == axes[0] else '')
    ax.set_title(f'k={k_val}')
    ax.axhline(0, color='black', linewidth=0.8)
    if ax == axes[0]:
        ax.legend(title='Method')

fig.suptitle('Absolute Recall Gain vs Popularity Baseline', y=1.02)
fig.text(0.99, 0.01, 'Source: experiment_results.parquet', ha='right', fontsize=8, color='gray')
plt.tight_layout()
plt.show()

# Summary table: recall gain by fraction and k
pivot_gain = gain_agg.pivot_table(
    index=['frac_label', 'method_label'],
    columns='k',
    values='recall_gain',
).round(4)
pivot_gain.index.names = ['n_obs', 'method']

method_order_no_pop = [METHOD_LABELS[m] for m in non_pop_methods]
pivot_gain = pivot_gain.reindex(
    pd.MultiIndex.from_product(
        [[FRAC_LABELS[f] for f in N_OBS_FRACTIONS], method_order_no_pop],
        names=['n_obs', 'method'],
    )
)
print('\nMean absolute recall gain over Popularity baseline')
display(pivot_gain)

## 11. Key Findings

**Interpretação dos resultados — preencha após rodar o experimento.**

- **LocalAgg vs Popularity**: a estrutura de 1-hop do grafo produz ganho mensurável sobre o baseline sem contexto. O ganho cresce com a fração observada porque mais itens seed ativam mais arestas diretas.
- **PPR vs LocalAgg**: a propagação multi-hop adiciona ganho sobre o LocalAgg. Espera-se ganho mais pronunciado em cestas menores (25% observado), onde os vizinhos diretos são poucos.
- **Saturação por fração**: comparar os ganhos de recall entre 25%, 50% e 75% revela a partir de qual fração o sinal do grafo começa a saturar.
- **Trade-off k**: recall aumenta com k, mas precision cai. O valor ótimo de k depende de quantos itens a interface pode exibir.

In [ ]:
full = pd.read_parquet('../outputs/results/experiment_results_full.parquet')

# Average over repetitions per basket, then over all baskets
agg_full = (
    full
    .groupby(['basket_id', 'n_obs', 'method', 'k'])[['precision', 'recall']]
    .mean()
    .groupby(['n_obs', 'method', 'k'])[['precision', 'recall']]
    .mean()
    .reset_index()
)
agg_full['method_label'] = agg_full['method'].map(METHOD_LABELS)

print(f'n_obs values: {sorted(agg_full["n_obs"].unique())}')
print(f'k values:     {sorted(agg_full["k"].unique())}')
print(f'Methods:      {sorted(agg_full["method"].unique())}')
agg_full.head()